In [51]:
import json
import os
import re
import time
from datetime import datetime

import requests
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from webdriver_manager.chrome import ChromeDriverManager



OUT_FILE = "fragment_final.jsonl"  
HEADLESS = True
ONLY_NEW = True                    # se True, non riscrapa asset già presenti
MAX_ASSETS = None                   # None = tutti quelli presenti in homepage
SLEEP_BETWEEN_ASSETS = 0.4


HEADERS = {
    "user-agent": "Mozilla/5.0",
}


BASE = "https://fragment.com"


# ---------- utils ----------
def now_str():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def append_jsonl(path, obj):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")


def load_known_assets(jsonl_path):
    if not os.path.exists(jsonl_path):
        return set()
    s = set()
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                o = json.loads(line)
                a = o.get("Asset")
                if isinstance(a, str) and a:
                    s.add(a)
            except Exception:
                pass
    return s


def parse_price_value(text):
    """
    Dal DOM Fragment:
      - prezzi spesso sono "1,500" (senza "TON")
      - a volte "Transferred" -> None
    Ritorna float o None.
    """
    if not text:
        return None

    t = str(text).strip().lower()
    if "transferred" in t:
        return None

    # prende il primo blocco numerico
    m = re.search(r"[\d][\d\s,\.]*", text)
    if not m:
        return None

    num = m.group(0).strip().replace(" ", "")

    # 1,500 -> 1500 (virgola migliaia)
    if "," in num and "." not in num:
        parts = num.split(",")
        if len(parts) == 2 and len(parts[1]) == 3:
            num = parts[0] + parts[1]
        else:
            num = num.replace(",", ".")  # fallback virgola decimale

    # 1.500 -> 1500 (punto migliaia)
    if "." in num:
        parts = num.split(".")
        if len(parts) == 2 and len(parts[1]) == 3:
            num = parts[0] + parts[1]

    try:
        return float(num)
    except Exception:
        return None


def normalize_dt_from_time_tag(t):
    """
    Preferisce <time datetime="2025-08-07T18:27:00Z">
    -> "2025-08-07 18:27:00"
    """
    if t is None:
        return None

    dt = t.get("datetime")
    if isinstance(dt, str) and dt:
        dt = dt.replace("T", " ").replace("Z", "")
        dt = re.split(r"\+|-", dt)[0].strip()
        
        return dt

    return t.get_text(strip=True)


def wallet_text(a_tag):
    """
    Nel DOM:
      - se è un nome: <span class="short">advanced.t.me</span>
      - se è un address spezzato: head + tail
    """
    if not a_tag:
        return None

    short = a_tag.find("span", class_="short")
    if short:
        return short.get_text(strip=True)

    head = a_tag.find("span", class_="head")
    tail = a_tag.find("span", class_="tail")
    if head and tail:
        return (head.get_text(strip=True) + tail.get_text(strip=True)).strip()

    return a_tag.get_text(" ", strip=True)


# ---------- selenium  ----------
def build_driver():
    options = webdriver.ChromeOptions()
    if HEADLESS:
        options.add_argument("--headless=new")

    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    
    options.add_argument("--disable-renderer-backgrounding")
    options.add_argument("--disable-background-timer-throttling")
    options.add_argument("--log-level=3")
    options.add_experimental_option("excludeSwitches", ["enable-logging"])

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    driver.set_page_load_timeout(60)
    return driver


def safe_get(driver, url, tries=3):
    for i in range(tries):
        try:
            driver.get(url)
            WebDriverWait(driver, 20).until(
                lambda d: d.execute_script("return document.readyState") == "complete"
            )
            return True
        except Exception as e:
            print(f"[WARN] get failed ({i+1}/{tries}): {e}")
            time.sleep(1.0)
    return False


def click_load_more(driver, class_name, max_clicks=200):
    """
    Bottoni:
      - js-load-more-orders (bids)
      - js-load-more-owners (ownership)
    Se la sezione non esiste o non c’è storico/offerte, il bottone può mancare.
    """
    for _ in range(max_clicks):
        try:
            btn = driver.find_element(By.CLASS_NAME, class_name)
            if not btn.is_displayed():
                break
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
            driver.execute_script("arguments[0].click();", btn)
            time.sleep(0.25)
        except Exception:
            break


# ---------- DOM parsing ----------
def homepage_assets():
    r = requests.get(BASE + "/", headers=HEADERS, timeout=30)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")
    rows = soup.find_all("tr", class_="tm-row-selectable")

    out = []
    for row in rows:
        username_tag = row.find("div", class_="table-cell-value tm-value")
        if not username_tag:
            continue

        asset = username_tag.get_text(strip=True)  # "@name"

        resale_tag = row.find("div", class_="table-cell-status-thin")
        resale = "yes" if resale_tag else "no"

        out.append({"Asset": asset, "Resale": resale})

        if MAX_ASSETS is not None and len(out) >= MAX_ASSETS:
            break

    return out


def find_section_by_h3(soup, title_text):
    """
    Cerca:
      <section class="tm-section clearfix"> ... <h3 class="tm-section-header-text">Bid History</h3>
    """
    for sec in soup.find_all("section", class_="tm-section clearfix"):
        h3 = sec.find("h3", class_="tm-section-header-text")
        if h3 and h3.get_text(strip=True).lower() == title_text.lower():
            return sec
    return None


def parse_table_rows_bid(section):
    """
    Bid History: per ogni riga:
      - col 0: price (div.table-cell-value ... icon-ton) -> numero
      - col 1: time
      - col 2: bidder wallet
    """
    out = []
    if not section:
        return out

    table = section.find("table")
    if not table:
        return out

    tbody = table.find("tbody")
    if not tbody:
        return out

    for tr in tbody.find_all("tr"):
        tds = tr.find_all("td")
        if len(tds) < 3:
            continue

        price_div = tds[0].find("div", class_=re.compile(r"table-cell-value"))
        time_tag = tds[1].find("time")
        wallet_a = tds[2].find("a", class_="tm-wallet")

        price = parse_price_value(price_div.get_text(" ", strip=True) if price_div else None)
        date = normalize_dt_from_time_tag(time_tag)
        bidder = wallet_text(wallet_a)

        out.append({"Bidder": bidder, "Date of Bid": date, "Price": price})

    return out


def parse_table_rows_ownership(section):
    """
    Ownership History: per ogni riga:
      - col 0: sale price (può essere "Transferred") -> None
      - col 1: time
      - col 2: owner wallet
    """
    out = []
    if not section:
        return out

    table = section.find("table")
    if not table:
        return out

    tbody = table.find("tbody")
    if not tbody:
        return out

    for tr in tbody.find_all("tr"):
        tds = tr.find_all("td")
        if len(tds) < 3:
            continue

        price_div = tds[0].find("div", class_=re.compile(r"table-cell-value"))
        time_tag = tds[1].find("time")
        wallet_a = tds[2].find("a", class_="tm-wallet")

        price = parse_price_value(price_div.get_text(" ", strip=True) if price_div else None)
        date = normalize_dt_from_time_tag(time_tag)
        owner = wallet_text(wallet_a)

        out.append({"Owner": owner, "Date of Sale": date, "Price": price})

    return out


def parse_highest_bid(soup):
    """
    Nel DOM c'è:
      <div class="tm-section-bid-info"> <table> ... <td> <div class="table-cell-value ... icon-ton">1,500</div>
    """
    box = soup.find("div", class_="tm-section-bid-info")
    if not box:
        return None

    td = box.find("td")
    if not td:
        return None

    v = td.find("div", class_=re.compile(r"table-cell-value"))
    return parse_price_value(v.get_text(" ", strip=True) if v else None)


def scrape_asset_page(driver, asset):
    slug = asset.lstrip("@")
    url = f"{BASE}/username/{slug}"

    if not safe_get(driver, url):
        return None

    # Carica tutto se ci sono bottoni "load more"
    click_load_more(driver, "js-load-more-orders")
    click_load_more(driver, "js-load-more-owners")

    soup = BeautifulSoup(driver.page_source, "html.parser")

    bid_sec = find_section_by_h3(soup, "Bid History")
    own_sec = find_section_by_h3(soup, "Ownership History")

    bids = parse_table_rows_bid(bid_sec)
    ownership = parse_table_rows_ownership(own_sec)

    highest_bid = parse_highest_bid(soup)

    rec = {
        "Asset": "@" + slug,
        "Scrape date": now_str(),
        "Ownership history": ownership,
        "Bids": bids,
    }

    if highest_bid is not None:
        rec["Highest bid"] = highest_bid

    return rec


def main():
    known = load_known_assets(OUT_FILE) if ONLY_NEW else set()
    assets = homepage_assets()

    if not assets:
        print("Nessun asset trovato in homepage.")
        return

    driver = build_driver()
    try:
        for item in assets:
            asset = item["Asset"]

            if ONLY_NEW and asset in known:
                continue

            rec = scrape_asset_page(driver, asset)
            if rec is None:
                print("[WARN] skip:", asset)
                continue

           
            rec["Resale"] = item.get("Resale")

            append_jsonl(OUT_FILE, rec)
            known.add(asset)

            print("Saved:", asset, "| bids:", len(rec["Bids"]), "| ownership:", len(rec["Ownership history"]))
            time.sleep(SLEEP_BETWEEN_ASSETS)

    finally:
        driver.quit()


if __name__ == "__main__":
    main()

In [21]:
import pandas as pd
df = pd.read_json("fragment_final.jsonl", lines=True)
df

,Asset,Scrape date,Ownership history,Bids,Highest bid,Resale
0,@singapore,2026-01-16 16:40:29,[],[{'Bidder': 'UQCUmsKJ9lBZhuHwS6bxcjUO2ddqMVnQy...,15000.0,no
1,@river,2026-01-16 16:40:30,[],[{'Bidder': 'UQDwZBi6t0w3xXSoEm2ZAqrXJaAwJJrSb...,5555.0,no
2,@trial,2026-01-16 16:40:31,[{'Owner': 'UQDRhSGR_e_QyBCNsZhet8VVSJYj1HAHOx...,"[{'Bidder': 'advanced.t.me', 'Date of Bid': 'M...",1500.0,yes
3,@baaa,2026-01-16 16:40:31,[],[{'Bidder': 'UQDt88M4PhcDUAJhKPs5tfwlcmSSjCToW...,5050.0,no
4,@bb9y,2026-01-16 16:40:32,[],[{'Bidder': 'UQAMuJnI_dCt0PDqmEaUeHvEcqIB5xALK...,5050.0,no
...,...,...,...,...,...,...
1095,@wmwww,2026-01-16 17:04:18,[],"[{'Bidder': 'bodde.t.me', 'Date of Bid': 'Jan ...",104.0,no
1096,@xa666,2026-01-16 17:04:19,[],[{'Bidder': 'UQBKGLTEUwrrnyydx-PAAi0flyQvYpavD...,104.0,no
1097,@xiaoguoding,2026-01-16 17:04:20,[],[{'Bidder': 'UQA5FvIIIN-nRHTLgMlibuXWpHLqdidDC...,104.0,no
1098,@xiaomanyc,2026-01-16 17:04:20,[],"[{'Bidder': 'delicate.ton', 'Date of Bid': 'Ja...",104.0,no
